# Punchbowl — Demand Rank vs Weather
Pairs Punchbowl's mean relative demand rank with temperature, dewpoint and relative humidity from weather_rank.

# 1. setup

In [ ]:
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")
%cd /home/565/pv3484/aus_substation_electricity
!pwd

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# 2. load data

In [ ]:
demand_rank = pd.read_csv("/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv")
weather_rank = pd.read_csv("/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v3/weather_holiday_block_means_with_codes.csv")

# Standardise holiday names
demand_rank["holiday_group"] = demand_rank["holiday_group"].replace({"Monarch's Birthday": "Queen's Birthday"})
weather_rank["holiday_group"] = weather_rank["holiday_group"].replace({"Monarch's Birthday": "Queen's Birthday"})

print("demand_rank shape:", demand_rank.shape)
print("weather_rank shape:", weather_rank.shape)

# 3. filter punchbowl and merge 

In [ ]:
station_code = 'PUNCH'
station_full_name = 'Punchbowl'

df_d = demand_rank[demand_rank['station_code'] == station_code].copy()
df_w = weather_rank[weather_rank['station_code'] == station_code].copy()

paired = pd.merge(
    df_d, df_w,
    on=['date', 'holiday_group', 'station_code'],
    how='inner',
    suffixes=('', '_w')
)

paired['is_weekend'] = paired.get('is_weekend', paired.get('is_weekend_w'))
paired['is_holiday'] = paired.get('is_holiday', paired.get('is_holiday_w'))

print("paired shape:", paired.shape)
print(paired.head())

In [ ]:
# Check column names for weather variables
print([c for c in paired.columns if any(x in c for x in ['temp', 'dewpoint', 'relative_humidity'])])

# Plot Functions

In [ ]:
# --- Plot 1: Temperature (x) coloured by Dewpoint ---

def plot_temp_dewpoint_scatter(
    paired,
    holiday_name,
    station_code='PUNCH',
    station_full_name='Punchbowl',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    weekday_color = '#D8D8D8'
    weekend_color = 'skyblue'
    dp_cmap = plt.cm.RdYlGn
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    dp_cols_all = [f'{b}_dewpoint' for b in time_blocks if f'{b}_dewpoint' in df.columns]
    if not dp_cols_all:
        dp_cols_all = [f'dewpoint_{b}' for b in time_blocks if f'dewpoint_{b}' in df.columns]
    dp_global_min = df[dp_cols_all].min().min()
    dp_global_max = df[dp_cols_all].max().max()
    dp_norm = Normalize(vmin=dp_global_min, vmax=dp_global_max)

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]

        rank_col = f'{block}_mean_relative_rank'
        x_col    = f'{block}_temp' if f'{block}_temp' in df.columns else f'temp_{block}'
        dp_col   = f'{block}_dewpoint' if f'{block}_dewpoint' in df.columns else f'dewpoint_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or x_col not in df.columns or df[x_col].isna().all():
            ax.set_visible(False)
            continue

        wd = df[~df['is_weekend']]
        ax.scatter(wd[x_col], wd[rank_col], color=weekday_color, alpha=0.35, s=35, zorder=1)
        we = df[df['is_weekend']]
        ax.scatter(we[x_col], we[rank_col], color=weekend_color, alpha=0.45, s=35, zorder=1)

        holiday_rows = df[df['is_holiday']].copy()
        if dp_col in df.columns and not holiday_rows[dp_col].isna().all():
            colors = dp_cmap(dp_norm(holiday_rows[dp_col].values))
        else:
            colors = ['#AAAAAA'] * len(holiday_rows)

        ax.scatter(holiday_rows[x_col], holiday_rows[rank_col],
                   c=colors, s=120, edgecolor='black', linewidth=0.7, zorder=2)

        ax.set_xlabel('Temperature (°C)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)
        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Temperature vs Mean Relative Rank  |  Holiday coloured by Dewpoint',
        fontsize=16, y=1.05, linespacing=1.3
    )

    weekday_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekday_color, markersize=8, label='Weekday')
    weekend_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekend_color, markersize=8, label='Weekend')
    fig.legend(handles=[weekday_handle, weekend_handle],
               title='Day Type', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.25, 0.02),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4, ncol=1, frameon=True)

    fig.subplots_adjust(top=0.86, bottom=0.22, left=0.07, right=0.98, wspace=0.18)

    sm = ScalarMappable(cmap=dp_cmap, norm=dp_norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.38, 0.04, 0.4, 0.04])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend='neither')
    cbar.set_label('Dewpoint (°C)', fontsize=12)
    cbar.ax.tick_params(labelsize=12)

    return fig

In [ ]:
# --- Plot 2: Relative Humidity (x) coloured by Year (tab20) ---

def plot_rh_year_scatter(
    paired,
    holiday_name,
    station_code='PUNCH',
    station_full_name='Punchbowl',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    df['year'] = pd.to_datetime(df['date']).dt.year
    years = sorted(df['year'].unique())
    year_palette = plt.cm.tab20.colors
    year_color_map = {yr: year_palette[i % len(year_palette)] for i, yr in enumerate(years)}

    weekday_color = '#D8D8D8'
    weekend_color = 'skyblue'
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]

        rank_col = f'{block}_mean_relative_rank'
        rh_col   = f'{block}_relative_humidity' if f'{block}_relative_humidity' in df.columns else f'relative_humidity_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or rh_col not in df.columns or df[rh_col].isna().all():
            ax.set_visible(False)
            continue

        wd = df[~df['is_weekend']]
        ax.scatter(wd[rh_col], wd[rank_col], color=weekday_color, alpha=0.35, s=35, zorder=1)
        we = df[df['is_weekend']]
        ax.scatter(we[rh_col], we[rank_col], color=weekend_color, alpha=0.45, s=35, zorder=1)

        holiday_rows = df[df['is_holiday']].copy()
        colors = [year_color_map[yr] for yr in holiday_rows['year']]
        ax.scatter(holiday_rows[rh_col], holiday_rows[rank_col],
                   c=colors, s=120, edgecolor='black', linewidth=0.7, zorder=2)

        ax.set_xlabel('Relative Humidity (%)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)
        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Relative Humidity vs Mean Relative Rank  |  Holiday coloured by Year',
        fontsize=16, y=1.05, linespacing=1.3
    )

    weekday_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekday_color, markersize=8, label='Weekday')
    weekend_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekend_color, markersize=8, label='Weekend')
    fig.legend(handles=[weekday_handle, weekend_handle],
               title='Day Type', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.25, 0.02),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4, ncol=1, frameon=True)

    year_handles = [mpatches.Patch(color=year_color_map[yr], label=str(yr)) for yr in years]
    fig.legend(handles=year_handles,
               title='Year', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.6, 0.02),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4,
               ncol=min(len(years), 7), frameon=True)

    fig.subplots_adjust(top=0.86, bottom=0.22, left=0.07, right=0.98, wspace=0.18)

    return fig

In [ ]:
# --- Plot 3: Relative Humidity (x) coloured by Temperature ---

def plot_rh_temp_scatter(
    paired,
    holiday_name,
    station_code='PUNCH',
    station_full_name='Punchbowl',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    weekday_color = '#D8D8D8'
    weekend_color = 'skyblue'
    temp_cmap = plt.cm.YlOrRd
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    temp_cols_all = [f'{b}_temp' for b in time_blocks if f'{b}_temp' in df.columns]
    if not temp_cols_all:
        temp_cols_all = [f'temp_{b}' for b in time_blocks if f'temp_{b}' in df.columns]
    temp_global_min = df[temp_cols_all].min().min()
    temp_global_max = df[temp_cols_all].max().max()
    temp_norm = Normalize(vmin=temp_global_min, vmax=temp_global_max)

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]

        rank_col = f'{block}_mean_relative_rank'
        rh_col   = f'{block}_relative_humidity' if f'{block}_relative_humidity' in df.columns else f'relative_humidity_{block}'
        temp_col = f'{block}_temp' if f'{block}_temp' in df.columns else f'temp_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or rh_col not in df.columns or df[rh_col].isna().all():
            ax.set_visible(False)
            continue

        wd = df[~df['is_weekend']]
        ax.scatter(wd[rh_col], wd[rank_col], color=weekday_color, alpha=0.35, s=35, zorder=1)
        we = df[df['is_weekend']]
        ax.scatter(we[rh_col], we[rank_col], color=weekend_color, alpha=0.45, s=35, zorder=1)

        holiday_rows = df[df['is_holiday']].copy()
        if temp_col in df.columns and not holiday_rows[temp_col].isna().all():
            colors = temp_cmap(temp_norm(holiday_rows[temp_col].values))
        else:
            colors = ['#AAAAAA'] * len(holiday_rows)

        ax.scatter(holiday_rows[rh_col], holiday_rows[rank_col],
                   c=colors, s=120, edgecolor='black', linewidth=0.7, zorder=2)

        ax.set_xlabel('Relative Humidity (%)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)
        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Relative Humidity vs Mean Relative Rank  |  Holiday coloured by Temperature',
        fontsize=16, y=1.05, linespacing=1.3
    )

    weekday_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekday_color, markersize=8, label='Weekday')
    weekend_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekend_color, markersize=8, label='Weekend')
    fig.legend(handles=[weekday_handle, weekend_handle],
               title='Day Type', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.25, 0.02),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4, ncol=1, frameon=True)

    fig.subplots_adjust(top=0.86, bottom=0.22, left=0.07, right=0.98, wspace=0.18)

    sm = ScalarMappable(cmap=temp_cmap, norm=temp_norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.38, 0.04, 0.4, 0.04])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend='neither')
    cbar.set_label('Temperature (°C)', fontsize=12)
    cbar.ax.tick_params(labelsize=12)

    return fig

# Test plots

In [ ]:
temp_cols = [c for c in paired.columns if 'temp' in c and 'dewpoint' not in c]
print(paired[temp_cols].min().min())
print(paired[temp_cols].max().max())

In [ ]:
test_holiday = "Christmas and Boxing Day"
print(f"Testing with: {test_holiday}")

fig1 = plot_temp_dewpoint_scatter(paired, test_holiday)
plt.show()

fig2 = plot_rh_year_scatter(paired, test_holiday)
plt.show()

fig3 = plot_rh_temp_scatter(paired, test_holiday)
plt.show()

# 6. saving

In [ ]:
save_dir = "/home/565/pv3484/aus_substation_electricity/data/figures/multi_dimension_mean_rank/seminar_punchbowl"

plot_types = {
    "temp_dewpoint": plot_temp_dewpoint_scatter,
    "rh_year":       plot_rh_year_scatter,
    "rh_temp":       plot_rh_temp_scatter,
}

for plot_type, plot_func in plot_types.items():
    folder = os.path.join(save_dir, plot_type)
    os.makedirs(folder, exist_ok=True)

    for holiday in paired["holiday_group"].dropna().unique():
        fig = plot_func(paired, holiday)
        if fig is not None:
            holiday_slug = holiday.replace(" ", "_").replace("'", "")
            save_path = os.path.join(folder, f"{holiday_slug}_{station_code}.png")
            fig.savefig(save_path, bbox_inches="tight", dpi=150)
            plt.close(fig)
            print(f"Saved: {plot_type}/{holiday_slug}")

print("Done.")

# Layering plots

## 1. temp vs dewpoint

### Weekdays only

In [ ]:
# --- Temp/Dewpoint: weekday only ---
def plot_temp_dewpoint_scatter_weekdays_punch(
    paired,
    holiday_name,
    station_code='PUNCH',
    station_full_name='Punchbowl',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    df = df[~df['is_weekend'] & ~df['is_holiday']]
    if df.empty:
        print(f'No weekday data for {holiday_name}')
        return None

    weekday_color = '#D8D8D8'
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]
        rank_col = f'{block}_mean_relative_rank'
        x_col    = f'{block}_temp' if f'{block}_temp' in df.columns else f'temp_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or x_col not in df.columns or df[x_col].isna().all():
            ax.set_visible(False)
            continue

        ax.scatter(df[x_col], df[rank_col],
                   color=weekday_color, s=60, edgecolor='black',
                   linewidth=0.4, zorder=2, alpha=0.8)

        ax.set_xlabel('Temperature (°C)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)

        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Temperature vs Mean Relative Rank  |  Weekdays only',
        fontsize=16, y=1.05, linespacing=1.3
    )

    fig.subplots_adjust(top=0.86, bottom=0.10, left=0.07, right=0.98, wspace=0.18)
    return fig

### Weekdays + weekends

In [ ]:
# --- Temp/Dewpoint: weekday + weekend, no public holiday ---
def plot_temp_dewpoint_scatter_non_holiday_punch(
    paired,
    holiday_name,
    station_code='PUNCH',
    station_full_name='Punchbowl',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    df = df[~df['is_holiday']]
    if df.empty:
        print(f'No non-holiday data for {holiday_name}')
        return None

    weekday_color = '#D8D8D8'
    weekend_color = 'skyblue'
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]
        rank_col = f'{block}_mean_relative_rank'
        x_col    = f'{block}_temp' if f'{block}_temp' in df.columns else f'temp_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or x_col not in df.columns or df[x_col].isna().all():
            ax.set_visible(False)
            continue

        wd = df[~df['is_weekend']]
        we = df[df['is_weekend']]

        ax.scatter(wd[x_col], wd[rank_col], color=weekday_color, s=60,
                   edgecolor='black', linewidth=0.4, zorder=1, alpha=0.8)
        ax.scatter(we[x_col], we[rank_col], color=weekend_color, s=60,
                   edgecolor='black', linewidth=0.4, zorder=2, alpha=0.8)

        ax.set_xlabel('Temperature (°C)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)

        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Temperature vs Mean Relative Rank  |  Excl. Public Holidays',
        fontsize=16, y=1.05, linespacing=1.3
    )

    weekday_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekday_color,
                                markeredgecolor='black', markeredgewidth=0.4, markersize=8, label='Weekday')
    weekend_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekend_color,
                                markeredgecolor='black', markeredgewidth=0.4, markersize=8, label='Weekend')
    fig.legend(handles=[weekday_handle, weekend_handle],
               title='Day Type', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.10, 0.06),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4, ncol=1, frameon=True)

    fig.subplots_adjust(top=0.86, bottom=0.18, left=0.07, right=0.98, wspace=0.18)
    return fig

## 2. relative humidity vs temp

### Weekdays only

In [ ]:
# --- RH/Temp: weekday only ---
def plot_rh_temp_scatter_weekdays_punch(
    paired,
    holiday_name,
    station_code='PUNCH',
    station_full_name='Punchbowl',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    df = df[~df['is_weekend'] & ~df['is_holiday']]
    if df.empty:
        print(f'No weekday data for {holiday_name}')
        return None

    weekday_color = '#D8D8D8'
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]
        rank_col = f'{block}_mean_relative_rank'
        rh_col   = f'{block}_relative_humidity' if f'{block}_relative_humidity' in df.columns else f'relative_humidity_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or rh_col not in df.columns or df[rh_col].isna().all():
            ax.set_visible(False)
            continue

        ax.scatter(df[rh_col], df[rank_col],
                   color=weekday_color, s=60, edgecolor='black',
                   linewidth=0.4, zorder=2, alpha=0.8)

        ax.set_xlabel('Relative Humidity (%)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)

        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Relative Humidity vs Mean Relative Rank  |  Weekdays only',
        fontsize=16, y=1.05, linespacing=1.3
    )

    fig.subplots_adjust(top=0.86, bottom=0.10, left=0.07, right=0.98, wspace=0.18)
    return fig

### Weekdays + weekends

In [ ]:
# --- RH/Temp: weekday + weekend, no public holiday ---
def plot_rh_temp_scatter_non_holiday_punch(
    paired,
    holiday_name,
    station_code='PUNCH',
    station_full_name='Punchbowl',
    time_blocks=None
):
    if time_blocks is None:
        time_blocks = ['00_04', '04_10', '10_15', '15_20', '20_24']

    df = paired[paired['holiday_group'] == holiday_name].copy()
    if df.empty:
        print(f'No data for {holiday_name}')
        return None

    df = df[~df['is_holiday']]
    if df.empty:
        print(f'No non-holiday data for {holiday_name}')
        return None

    weekday_color = '#D8D8D8'
    weekend_color = 'skyblue'
    block_titles = ['12am-4am', '4am-10am', '10am-3pm', '3pm-8pm', '8pm-12am']

    fig, axes = plt.subplots(1, len(time_blocks), figsize=(4.5 * len(time_blocks), 5.0), sharey=False)

    for col_idx, block in enumerate(time_blocks):
        ax = axes[col_idx]
        rank_col = f'{block}_mean_relative_rank'
        rh_col   = f'{block}_relative_humidity' if f'{block}_relative_humidity' in df.columns else f'relative_humidity_{block}'

        ax.set_title(block_titles[col_idx], fontsize=14, pad=10)

        if rank_col not in df.columns or rh_col not in df.columns or df[rh_col].isna().all():
            ax.set_visible(False)
            continue

        wd = df[~df['is_weekend']]
        we = df[df['is_weekend']]

        ax.scatter(wd[rh_col], wd[rank_col], color=weekday_color, s=60,
                   edgecolor='black', linewidth=0.4, zorder=1, alpha=0.8)
        ax.scatter(we[rh_col], we[rank_col], color=weekend_color, s=60,
                   edgecolor='black', linewidth=0.4, zorder=2, alpha=0.8)

        ax.set_xlabel('Relative Humidity (%)', fontsize=13)
        if col_idx == 0:
            ax.set_ylabel('Mean Relative Rank', fontsize=13)
        else:
            ax.set_ylabel('')
            ax.tick_params(axis='y', labelleft=False)

        ax.tick_params(axis='both', labelsize=10)
        ax.margins(0.05)

    fig.suptitle(
        f'{holiday_name} — {station_full_name}\n'
        f'Relative Humidity vs Mean Relative Rank  |  Excl. Public Holidays',
        fontsize=16, y=1.05, linespacing=1.3
    )

    weekday_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekday_color,
                                markeredgecolor='black', markeredgewidth=0.4, markersize=8, label='Weekday')
    weekend_handle = plt.Line2D([], [], marker='o', linestyle='', color=weekend_color,
                                markeredgecolor='black', markeredgewidth=0.4, markersize=8, label='Weekend')
    fig.legend(handles=[weekday_handle, weekend_handle],
               title='Day Type', title_fontsize=12, fontsize=12,
               loc='center', bbox_to_anchor=(0.10, 0.06),
               borderpad=0.4, labelspacing=0.3, handletextpad=0.4, ncol=1, frameon=True)

    fig.subplots_adjust(top=0.86, bottom=0.18, left=0.07, right=0.98, wspace=0.18)
    return fig

## loop and save

#### Weekday only

In [ ]:
# ── Saving loop ─────────────────────────────────────────────────────────────

def save_layered_scatters_punch(paired, holiday_names, station_code='PUNCH', station_full_name='Punchbowl'):

    base_dir = '/home/565/pv3484/aus_substation_electricity/pia_notebooks/NSW_data/relative_ranking/seminar_MOSMA_PUNCH/layered_scatters'

    weekday_dir     = os.path.join(base_dir, 'weekday_only')
    non_holiday_dir = os.path.join(base_dir, 'weekday_weekend')

    os.makedirs(weekday_dir, exist_ok=True)
    os.makedirs(non_holiday_dir, exist_ok=True)

    for holiday in holiday_names:
        print(f'Processing: {holiday}')
        safe_name = holiday.replace(' ', '_').replace('/', '-')

        # --- weekday only ---
        fig = plot_temp_dewpoint_scatter_weekdays_punch(
            paired, holiday, station_code=station_code, station_full_name=station_full_name
        )
        if fig is not None:
            path = os.path.join(weekday_dir, f'{safe_name}_temp_dewpoint_weekday_PUNCH.png')
            fig.savefig(path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f'  Saved: {path}')

        fig = plot_rh_temp_scatter_weekdays_punch(
            paired, holiday, station_code=station_code, station_full_name=station_full_name
        )
        if fig is not None:
            path = os.path.join(weekday_dir, f'{safe_name}_rh_temp_weekday_PUNCH.png')
            fig.savefig(path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f'  Saved: {path}')

        # --- weekday + weekend (no public holiday) ---
        fig = plot_temp_dewpoint_scatter_non_holiday_punch(
            paired, holiday, station_code=station_code, station_full_name=station_full_name
        )
        if fig is not None:
            path = os.path.join(non_holiday_dir, f'{safe_name}_temp_dewpoint_non_holiday_PUNCH.png')
            fig.savefig(path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f'  Saved: {path}')

        fig = plot_rh_temp_scatter_non_holiday_punch(
            paired, holiday, station_code=station_code, station_full_name=station_full_name
        )
        if fig is not None:
            path = os.path.join(non_holiday_dir, f'{safe_name}_rh_temp_non_holiday_PUNCH.png')
            fig.savefig(path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f'  Saved: {path}')

    print('Done.')

In [ ]:
holiday_names = paired['holiday_group'].unique()
save_layered_scatters_punch(paired, holiday_names)